In [ ]:
import os

import numpy as np
from dotenv import load_dotenv
from nhp.model.params import load_sample_params

from nhp.databricks.national import DatabricksNational
from nhp.databricks.run import Run

load_dotenv()

In [ ]:
params = load_sample_params(dataset="national", model_runs=4, app_version="dev")
data_path = f"{os.getenv('DATA_PATH')}/v6.0.0"

In [ ]:
sample_rate = 0.0001
r = Run(params, DatabricksNational, data_path, seed=1, sample_rate=sample_rate)
r.go()

In [ ]:
(
    r.results["default"]
    .assign(value=lambda x: x["value"] * 1 / sample_rate)
    .assign(run_type=lambda x: np.where(x["model_run"] == 0, "baseline", "principal"))
    .groupby(["pod", "measure", "run_type"], as_index=False)["value"]
    .mean()
    .pivot(index=["pod", "measure"], columns="run_type", values="value")
    .reset_index()
    .assign(diff=lambda x: x["principal"] - x["baseline"])
)